In [1]:
!pip install pydantic==2.12.3 -q
!pip install openai-agents -q
!pip install openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.9/468.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 11.0 MB/s eta 0:00:00


In [2]:
import os
import requests
from openai import OpenAI
from typing import List, Optional
from pydantic import BaseModel, Field

In [3]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY']=OPENAI_API_KEY

In [4]:
def render_transcript(d: dict) -> str:
    """
    Renders the transcript data from a dictionary into a formatted string.

    Args:
        d: A dictionary containing the transcript data. Expected keys are 'symbol',
           'quarter', and 'transcript'. The 'transcript' key should contain a list
           of dictionaries, each with 'speaker', 'title' (optional), and 'content'.

    Returns:
        A formatted string representation of the transcript.
    """
    header_info = f"Symbol: {d.get('symbol','')} | Quarter: {d.get('quarter','')}"
    transcript_lines = [header_info, "TRANSCRIPT:"]
    for index, transcript_entry in enumerate(d.get("transcript", [])):
        speaker_name = transcript_entry.get("speaker", "Unknown")
        speaker_title = transcript_entry.get("title")
        utterance_text = (transcript_entry.get("content") or "").strip()
        speaker_tag = f"{speaker_name} ({speaker_title})" if speaker_title else speaker_name
        transcript_lines.append(f"[{index:04d}] {speaker_tag}: {utterance_text}")
    return "\n".join(transcript_lines)

In [5]:
class EarningsCallInsights(BaseModel):
    """
    Output estructurado con insights clave en español extraídos de una earnings call.
    """
    symbol: Optional[str] = Field(description="Ticker de la compañía, ej. AAPL, AMZN, MSFT, etc")
    sentiment: Optional[str] = Field(description="Sentimiento general: Muy bajista / Bajista / Alcista / Muy alcista")
    summary: str = Field(description="Resumen ejecutivo de la llamada en español (párrafo de 3 líneas)")
    key_topics: List[str] = Field(default_factory=list, description="Temas estratégicos principales, cado uno con un máximo de 4 palabras")
    guidance: List[str] = Field(default_factory=list, description="Guías o proyecciones futuras mencionadas, en español")
    numeric_highlights: List[str] = Field(default_factory=list, description="Cifras o métricas clave reportadas, en español")
    risks: List[str] = Field(default_factory=list, description="Riesgos explícitos o implícitos discutidos, en español")
    catalysts: List[str] = Field(default_factory=list, description="Catalizadores futuros o eventos relevantes, en español")
    analyst_questions: List[str] = Field(default_factory=list, description="Top 3 preguntas destacadas de analistas, en español")
    unanswered_topics: List[str] = Field(default_factory=list, description="Temas abiertos o sin respuesta clara, en español")
    bullish_points: List[str] = Field(default_factory=list, description="Tesis alcistas derivadas de la llamada, en español")
    bearish_points: List[str] = Field(default_factory=list, description="Tesis bajistas derivadas de la llamada, en español")
    red_flags: List[str] = Field(default_factory=list, description="Alertas o señales negativas detectadas, en español")
    emotion: Optional[str] = Field(description="Emoción general: Optimismo / Incertidumbre / Preocupación / Entusiasmo / Frustración")

In [6]:
def get_earnings_call_insights(call_transcripts: dict) -> EarningsCallInsights:
    """
    Obtiene insights clave de una transcripción de earnings call utilizando un modelo de OpenAI.

    Args:
        transcript_text: El texto de la transcripción de la llamada.

    Returns:
        Un objeto diccionario derivado de EarningsCallInsights con los insights extraídos.
    """
    client = OpenAI()
    transcript_text = render_transcript(call_transcripts)
    response = client.chat.completions.parse(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": "Eres un experto Analista Financiero. Devuelve SOLO un JSON válido que siga exactamente el esquema de EarningsCallInsights. Salidas en español."},
            {"role": "user", "content": transcript_text},
        ],
        response_format=EarningsCallInsights,
    )
    insights = response.choices[0].message.parsed
    return insights.model_dump()

In [7]:
def get_call_transcripts(ticker: str, quarter: str) -> dict:
    client = OpenAI()
    alphavantage_api = userdata.get('ALPHAVANTAGE_API')
    url_transcripts = f'https://www.alphavantage.co/query?function=EARNINGS_CALL_TRANSCRIPT&symbol={ticker}&quarter={quarter}&apikey={alphavantage_api}'
    response_transcripts = requests.get(url_transcripts)
    call_transcripts = response_transcripts.json()
    insights = get_earnings_call_insights(call_transcripts=call_transcripts)
    return insights

### OpenAI Agents SDK

In [8]:
from agents import Agent, Runner, function_tool, trace

In [9]:
@function_tool
def get_call_transcripts(ticker: str, quarter: str) -> dict:
    client = OpenAI()
    alphavantage_api = userdata.get('ALPHAVANTAGE_API')
    url_transcripts = f'https://www.alphavantage.co/query?function=EARNINGS_CALL_TRANSCRIPT&symbol={ticker}&quarter={quarter}&apikey={alphavantage_api}'
    response_transcripts = requests.get(url_transcripts)
    call_transcripts = response_transcripts.json()
    insights = get_earnings_call_insights(call_transcripts=call_transcripts)
    return insights

In [11]:
earnings_agent = Agent(
    name="EarningsAgent",
    instructions="Retorna al usuario los hallazgos más relevantes de las llamadas con inversionistas de una conference call. Usa las herramientas a tu disposición cuando se solicite alguna llamada con inversionistas.",
    tools=[get_call_transcripts],
    model="gpt-5.4-mini",
)

In [12]:
instructions = "Insights de la llamada con inversionistas de NVDA del 2026Q2"

In [13]:
with trace("Earnings Call"):
    result = await Runner.run(earnings_agent, instructions)
    print(result.final_output)

Aquí van los **insights más relevantes** de la llamada con inversionistas de **NVIDIA (NVDA) 2026Q2**:

## Resumen ejecutivo
NVIDIA volvió a mostrar un trimestre **muy alcista**, con ingresos récord de **US$46.7 mil millones** y una demanda de IA que sigue **superando la oferta**. El motor principal sigue siendo **Data Center**, impulsado por **Blackwell**, **Blackwell Ultra** y **Networking**. La compañía también dejó claro que **Rubin** ya está en fábrica para el próximo ciclo, lo que refuerza la visibilidad de crecimiento estructural.

## Lo más importante
- **Data Center sigue liderando** el crecimiento, con fuerte tracción en IA de entrenamiento e inferencia.
- **Blackwell** continúa acelerando y Blackwell Ultra ya está ganando adopción.
- **Networking** se está volviendo un pilar clave: alcanzó **US$7.3 mil millones**, con crecimiento muy fuerte.
- La dirección mantiene una visión de largo plazo muy positiva por la expansión de:
  - **IA agente**
  - **IA soberana**
  - **Enterpr

Monitoreo:
[OpenAI Tracing](https://platform.openai.com/logs?api=traces)

Ejemplos:

* Insights de la llamada con inversionistas de AMZN del 2025Q4
* Insights de la llamada con inversionistas de GOOGL del 2025Q4
* Insights de la llamada con inversionistas del NVDA del 2026Q2
* Insights de la llamada con inversionistas de AMD del 2025Q4